# S09 - VIX term structure (volatility, late-data)

**Hypothesis.** Implied volatility is normally upward-sloping (1-month VIX below 3-month VIX3M).
When the curve inverts (backwardation), the market prices acute near-term stress; such episodes
cluster around crashes. Holding T-bills while VIX / VIX3M is high should avoid the worst days.

| Variant | Risk-on (hold S&P 500) when ... |
|---|---|
| ratio_lt_1.00 | previous day's VIX / VIX3M < 1.00 (contango) |
| ratio_lt_0.95 | previous day's VIX / VIX3M < 0.95 (stricter: de-risk earlier) |

**Inputs and point-in-time lags**

| Series | Source | Start | Lag used |
|---|---|---|---|
| VIX close | FRED `VIXCLS` (Cboe) | 1990 | previous trading day (+1 day) |
| VIX3M close | FRED `VXVCLS` (Cboe) | 2007-12 | previous trading day (+1 day) |

Daily check, next-close execution, 5 bp per switch. **Late-data protocol**: dev = data start..2013
(includes 2008), val 2014-2017, holdout 2018+, separate leaderboard, weaker evidence.

**Sources.** Cboe VIX / VIX3M methodology (https://www.cboe.com/tradable_products/vix/); Johnson
(2017), *Risk premia and the VIX term structure*, JFQA; FRED, https://fred.stlouisfed.org.

> Decision-support research only; never places orders.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
import bt_common as bc  # noqa: E402
import bt_long  # noqa: E402
import bt_periods as bp  # noqa: E402
import bt_pit  # noqa: E402
import bt_registry as br  # noqa: E402
import bt_stats as bs  # noqa: E402
import seed_registry  # noqa: E402
import stage2 as s2  # noqa: E402

bc.setup_plotly("cdn")
pd.set_option("display.width", 180)
TRIAL_ID, STRATEGY, NOTEBOOK = 'S09', 'vix_term_structure', 'volatility/vix_term_structure.ipynb'
PROTOCOL, REBALANCE = 'late', 'daily'

sample = bt_long.long_sample()
px, cash = sample["px"], sample["cash"]
bench = bp.benchmark_returns(px, cash)  # SMA200 monthly / B&H / 60-40 / T-bills, frozen settings
print("prices", px.index[0].date(), "->", px.index[-1].date(), "| registry trials so far:", br.trial_count())

prices 1934-02-01 -> 2026-10-19 | registry trials so far: 29


## 1. Data and point-in-time inputs

In [2]:
vix, vix3m = bt_long.fred_series("VIXCLS"), bt_long.fred_series("VXVCLS")
inputs = {"px": px, "vix": vix, "vix3m": vix3m}
print("VIX", vix.index[0].date(), "->", vix.index[-1].date(), "| VIX3M", vix3m.index[0].date(), "->", vix3m.index[-1].date())

VIX 1990-01-02 -> 2026-10-16 | VIX3M 2007-12-04 -> 2026-10-16


## 2. Signal and variant grid (fixed before any backtest)

In [3]:
def build_signal(inputs, thr=1.0):
    ratio = (inputs["vix"] / inputs["vix3m"]).dropna()
    ratio = bt_pit.daily_pit(ratio, inputs["px"].index, days=1)
    return (ratio < thr).astype(float).where(ratio.notna())


VARIANTS = {"ratio_lt_1.00": dict(thr=1.0), "ratio_lt_0.95": dict(thr=0.95)}
PARAMS_TEXT = {n: f"{n}; FRED VIXCLS/VXVCLS, lag 1 day; daily; next_close; 5bp" for n in VARIANTS}

In [4]:
ratio = (vix / vix3m).dropna()
fig = go.Figure(go.Scatter(x=ratio.index, y=ratio, name="VIX / VIX3M", line=dict(width=1)))
fig.add_hline(y=1.0, line_dash="dash", line_color="red")
fig.update_layout(title="VIX term-structure ratio (> 1 = backwardation)", height=340)
fig.show()
print(f"share of days in backwardation: {(ratio >= 1).mean():.1%}")

share of days in backwardation: 10.1%


## 3. Look-ahead checks

Every variant is recomputed with all inputs cut at past dates (monthly inputs: completed months only) and must equal the full-data signal on those dates (`stage2.truncation_test`). No backtest runs unless all pass.

In [5]:
N_CUTS = 30
trunc = {}
for name, params in VARIANTS.items():
    trunc[name] = s2.truncation_test(build_signal, inputs, n_cuts=N_CUTS, **params)
display(pd.DataFrame({k: v.row() for k, v in trunc.items()}).T)
bad = [k for k, v in trunc.items() if not v.passed]
assert not bad, f"look-ahead detected in variants {bad}: fix before any backtest"
print("all variants pass the truncation look-ahead test")

,cuts,mismatches,max_abs_diff,passed
ratio_lt_1.00,30,0,0.0,True
ratio_lt_0.95,30,0,0.0,True


all variants pass the truncation look-ahead test


## 4. Development-period sweep

Each variant is scored on the **dev** period only and registered as a trial. The best dev Sharpe is chosen; validation and holdout results are never used for the choice.

In [6]:
signals = {name: build_signal(inputs, **params) for name, params in VARIANTS.items()}
sweep, CHOSEN = s2.dev_sweep(px, cash, bench, signals, PARAMS_TEXT, trial_id=TRIAL_ID, strategy=STRATEGY,
                             notebook=NOTEBOOK, protocol=PROTOCOL, rebalance=REBALANCE)
display(sweep)
print("chosen on dev Sharpe only:", CHOSEN, "|", PARAMS_TEXT[CHOSEN])

,dev_sharpe,d_sma200,d_bh,time_in_market,switches_pa,dev_start
ratio_lt_1.00,0.468001,-0.410259,0.130635,0.844139,17.306521,2007-12-07
ratio_lt_0.95,0.722463,-0.155797,0.385097,0.664047,22.251241,2007-12-07


chosen on dev Sharpe only: ratio_lt_0.95 | ratio_lt_0.95; FRED VIXCLS/VXVCLS, lag 1 day; daily; next_close; 5bp


## 5. Frozen evaluation of the chosen variant

Frozen primary metric (`PLAN.md` section 2): beat SMA200 and buy & hold in dev **and** val, one-sided block-bootstrap p <= 0.10 vs SMA200 in both, deflated Sharpe >= 0.95 with the registry trial count. Holdout is confirmatory only.

In [7]:
signal = signals[CHOSEN]
runs, cmp, stats, res = s2.evaluate(px, cash, bench, signal, protocol=PROTOCOL, rebalance=REBALANCE)
display(cmp.round(3))
display(stats)
display(res.to_frame().round(3))
print(f"n_trials={res.n_trials} | DSR={res.dsr:.3f} | passes={res.passes} | holdout confirms={res.holdout_confirms}")
display(pd.Series(res.checks, name="check"))

,strategy,bh,d_bh,sma200,d_sma200,mix60,d_mix60
dev,0.722,0.337,0.385,0.878,-0.156,0.319,0.403
val,0.335,0.971,-0.636,0.771,-0.435,0.968,-0.633
holdout,0.559,0.675,-0.115,0.546,0.013,0.673,-0.113
full,0.571,0.563,0.009,0.693,-0.122,0.556,0.015


,CAGR,Vol,Sharpe,Sortino,Max DD,Calmar,Worst day,MaxDD peak,MaxDD recovered,Final x,Time in market,Switches,Switches / yr,Cost drag / yr
dev,0.085639,0.11921,0.722463,1.029074,-0.160857,0.532394,-0.032226,2011-04-29,2012-02-16,1.646292,0.664047,135,22.251241,0.011126
val,0.031799,0.098219,0.3352,0.447995,-0.21454,0.148218,-0.039386,2014-12-05,2017-11-03,1.133003,0.867925,68,17.046671,0.008523
holdout,0.090596,0.122188,0.559472,0.763079,-0.268896,0.336921,-0.058829,2022-01-03,2024-02-09,2.143963,0.779285,194,22.060554,0.01103
full,0.076232,0.116508,0.571286,0.787695,-0.268896,0.283501,-0.058829,2022-01-03,2024-02-09,3.999032,0.761012,397,21.042556,0.010521


,Sharpe,d_sma200,d_bh,p_vs_sma200
dev,0.722,-0.156,0.385,0.591
val,0.335,-0.435,-0.636,0.984
holdout,0.559,0.013,-0.115,NaN
full,0.571,-0.122,0.009,NaN


n_trials=31 | DSR=0.642 | passes=False | holdout confirms=True


dev_beats_sma200        False
dev_beats_bh             True
dev_p_le_0.1            False
val_beats_sma200        False
val_beats_bh            False
val_p_le_0.1            False
dsr_ge_0.95             False
holdout_d_sma200_pos     True
Name: check, dtype: bool

In [8]:
s2.plot_growth(runs, bench, f'{TRIAL_ID} {CHOSEN}', PROTOCOL).show()

## 6. Robustness: costs, placebo, peek test, decades

In [9]:
cs = bs.cost_sensitivity(px, signal, cash, rebalance=REBALANCE)
display(cs.round(3))
sma_full = bs.excess_sharpe(bench["sma200"].reindex(runs["full"].index), bench["tbill"].reindex(runs["full"].index))
be = bs.breakeven_cost_bps(px, signal, cash, bench_sharpe=sma_full, rebalance=REBALANCE)
print(f"break-even one-way cost vs SMA200 Sharpe over the same window ({sma_full:.3f}): {be:.1f} bp")
pl = bs.placebo_test(px, signal, cash, n_placebo=200, rebalance=REBALANCE, rng=np.random.default_rng(bs.SEED))
display(pd.Series(pl, name="placebo").to_frame())
peek = bt_pit.lookahead_test(px, signal, cash, rebalance=REBALANCE)
print(f"peek test: Sharpe by shift {peek.sharpe_by_shift.round(3).to_dict()} | gain {peek.peek_gain:+.3f}")
display(s2.decade_table(runs["full"], bench).style.format("{:.2f}"))

,Sharpe,CAGR,Max DD,turnover_pa
cost_bps,,,,
0,0.662,0.088,-0.261,21.084
5,0.571,0.076,-0.269,21.084
10,0.481,0.065,-0.279,21.084
25,0.209,0.032,-0.317,21.084
50,-0.239,-0.021,-0.540,21.084


break-even one-way cost vs SMA200 Sharpe over the same window (0.693): 0.0 bp


,placebo
stat,Sharpe
real,0.571286
placebo_mean,0.198207
placebo_q95,0.364503
quantile,1.0
p_one_sided,0.0
n_placebo,200
time_in_market,0.155021


peek test: Sharpe by shift {-21: 1.07, -5: 2.083, -1: 0.672, 0: 0.571, 1: 0.404, 5: 0.421, 21: 0.336} | gain +0.499


,Sharpe,Sharpe B&H,Sharpe SMA200,CAGR,CAGR B&H,CAGR SMA200,time in market
2000s,-0.02,-0.21,1.18,-0.00,-0.11,0.11,0.37
2010s,0.69,0.90,0.71,0.08,0.14,0.09,0.81
2020s,0.58,0.67,0.58,0.10,0.15,0.10,0.80


## 7. Register the result

In [10]:
note = ("passes" if res.passes else "fails") + f" frozen metric; chosen {CHOSEN} on dev; protocol {PROTOCOL}"
rows = s2.register_final(TRIAL_ID, STRATEGY, NOTEBOOK, PARAMS_TEXT[CHOSEN], res, note=note, header=seed_registry.HEADER)
display(rows)
print("registry trial count:", br.trial_count())

,trial_id,date,strategy,notebook,params,period,sharpe,d_sma200,d_bh,p_vs_sma200,dsr,n_trials,passes,status,note
73,S09-v1/dev,2026-10-20,vix_term_structure,volatility/vix_term_structure.ipynb,"ratio_lt_1.00; FRED VIXCLS/VXVCLS, lag 1 day; ...",dev,0.4680,-0.4103,0.1306,,,,,run,"dev-only sweep, variant ratio_lt_1.00"
74,S09-v2/dev,2026-10-20,vix_term_structure,volatility/vix_term_structure.ipynb,"ratio_lt_0.95; FRED VIXCLS/VXVCLS, lag 1 day; ...",dev,0.7225,-0.1558,0.3851,,,,,run,"dev-only sweep, variant ratio_lt_0.95"
75,S09/dev,2026-10-20,vix_term_structure,volatility/vix_term_structure.ipynb,"ratio_lt_0.95; FRED VIXCLS/VXVCLS, lag 1 day; ...",dev,0.7225,-0.1558,0.3851,0.5907,,31,False,fail,fails frozen metric; chosen ratio_lt_0.95 on d...
76,S09/val,2026-10-20,vix_term_structure,volatility/vix_term_structure.ipynb,"ratio_lt_0.95; FRED VIXCLS/VXVCLS, lag 1 day; ...",val,0.3352,-0.4355,-0.6361,0.9845,,31,False,fail,fails frozen metric; chosen ratio_lt_0.95 on d...
77,S09/holdout,2026-10-20,vix_term_structure,volatility/vix_term_structure.ipynb,"ratio_lt_0.95; FRED VIXCLS/VXVCLS, lag 1 day; ...",holdout,0.5595,0.0135,-0.1153,,,31,False,fail,fails frozen metric; chosen ratio_lt_0.95 on d...
78,S09/full,2026-10-20,vix_term_structure,volatility/vix_term_structure.ipynb,"ratio_lt_0.95; FRED VIXCLS/VXVCLS, lag 1 day; ...",full,0.5713,-0.1220,0.0085,,0.6424,31,False,fail,fails frozen metric; chosen ratio_lt_0.95 on d...


registry trial count: 31


## 8. Current signal

In [11]:
last = signal.dropna()
print(f"latest signal ({CHOSEN}) on {last.index[-1].date()}: equity weight {last.iloc[-1]:.2f}")
chk = bc.check_days(px.index, REBALANCE)
display(last[chk.reindex(last.index).fillna(False)].tail(6).to_frame("equity weight at decision"))
print("Research output only -- not a recommendation or an order.")

latest signal (ratio_lt_0.95) on 2026-10-19: equity weight 0.00


,equity weight at decision
Date,
2026-10-12,1.0
2026-10-13,1.0
2026-10-14,1.0
2026-10-15,1.0
2026-10-16,1.0
2026-10-19,0.0


Research output only -- not a recommendation or an order.


## 9. Evidence summary (computed)

In [12]:
for line in s2.evidence_lines(res, trunc[CHOSEN], peek, pl, be):
    print("*", line)

* Truncation look-ahead test: PASSED on 30 cut dates; peek test gain +0.499.
* dev: Sharpe 0.722 (vs SMA200 -0.156, vs B&H +0.385, p vs SMA200 0.591)
* val: Sharpe 0.335 (vs SMA200 -0.435, vs B&H -0.636, p vs SMA200 0.985)
* holdout: Sharpe 0.559 (vs SMA200 +0.013, vs B&H -0.115)
* Deflated Sharpe 0.642 with n_trials=31.
* Placebo: real 0.571 vs placebo 95th pct 0.365 (p 0.000); break-even cost vs SMA200 full 0.0 bp.
* Verdict: FAILS the frozen primary metric (failed checks: dev_beats_sma200, dev_p_le_0.1, val_beats_sma200, val_beats_bh, val_p_le_0.1, dsr_ge_0.95); holdout confirms.


## 10. Caveats

* Backwardation spells are short and sharp; with a one-day data lag plus next-close execution the rule
  often exits after the first big down day and re-enters after the rebound has started.
* Daily switching: costs matter, see the cost table.
* The ratio is observable only since 2007-12; one full bear market (2008) and two crashes (2020, 2025 if
  any) in sample: weak evidence.

## 11. Interpretation

* Late-data, weak evidence (2007-12 onward). The stricter `ratio_lt_0.95` won the dev sweep (0.72 vs 0.47
  for 1.00). In dev 2008-2013 it beats buy & hold clearly (+0.39 Sharpe, the 2008 crash) but **not SMA200**,
  which also sidestepped 2008 and scores 0.88 (p 0.59).
* Val 2014-2017: Sharpe 0.34 vs B&H 0.97 and SMA200 0.77 (p 0.98). In a calm, rising market short volatility
  spikes flip the rule out and back in after the rebound,
  at ~21 switches a year. Holdout 0.56, level with SMA200 (+0.01) but below B&H (0.67). **Fails**.
* Costs bite: Sharpe 0.66 at zero cost, 0.57 at 5 bp, 0.21 at 25 bp.
* The placebo p of 0.000 and the large peek gain (+0.50) show the ratio does carry information about the
  *next few days*; with a one-day data lag plus next-close execution most of it is gone before the
  trade is filled. Same-day execution on intraday data is outside this programme's no-look-ahead rules.